# Comparing Kubernetes Workload Types

A reference guide to Deployments, StatefulSets, DaemonSets, and Jobs — what each workload model is designed for, how they differ, and when to reach for one over the others.

## Purpose

Kubernetes offers several workload resource types that serve different scheduling and lifecycle needs. This notebook compares the four most commonly used types — Deployments, StatefulSets, DaemonSets, and Jobs — across their core characteristics, so that a practitioner can choose the right workload model for a given use case.

## When to use

| Workload Type | Best suited for | Avoid when |
|---|---|---|
| Deployment | Stateless applications that can be scaled horizontally and replaced at any time | The application requires stable network identities or persistent storage per replica |
| StatefulSet | Stateful applications that need stable network identities, ordered deployment, or persistent storage | The application is stateless and does not need ordered startup or stable identity |
| DaemonSet | Running a copy of a pod on every node (e.g., log collectors, monitoring agents) | Only a subset of nodes should run the workload, or the workload is stateless and horizontally scalable |
| Job | Batch processing that runs to completion, such as data migration or image processing | The workload is a long-running service that should not terminate after completing its task

## Steps

The following sections walk through each workload type with a minimal manifest and a brief explanation of the key fields.

### 1. Deployment

A Deployment manages a set of identical pods with no stable identity. It supports rolling updates and rollback, making it the default choice for stateless workloads.

In [ ]:
# last_verified: 2026-08-04 · k8s n/a
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-deployment
spec:
  replicas: 3
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      containers:
      - name: nginx
        image: nginx:1.25
        ports:
        - containerPort: 80

### 2. StatefulSet

A StatefulSet provides stable network identities (pod-0, pod-1, ...) and ordered deployment and scaling. Each pod can claim a dedicated PersistentVolumeClaim, making it suitable for stateful applications like databases.

In [ ]:
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: mysql
spec:
  serviceName: mysql
  replicas: 3
  selector:
    matchLabels:
      app: mysql
  template:
    metadata:
      labels:
        app: mysql
    spec:
      containers:
      - name: mysql
        image: mysql:8.0
        ports:
        - containerPort: 3306
        volumeMounts:
        - name: data
          mountPath: /var/lib/mysql
  volumeClaimTemplates:
- metadata:
    name: data
  spec:
    accessModes: ["ReadWriteOnce"]
    resources:
      requests:
        storage: 10Gi

### 3. DaemonSet

A DaemonSet ensures that a copy of a pod runs on every node in the cluster. It is commonly used for node-level agents such as log collectors or monitoring daemons.

In [ ]:
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: fluentd
spec:
  selector:
    matchLabels:
      app: fluentd
  template:
    metadata:
      labels:
        app: fluentd
    spec:
      containers:
      - name: fluentd
        image: fluentd:v1.16
        volumeMounts:
        - name: varlog
          mountPath: /var/log

### 4. Job

A Job creates one or more pods and ensures that a specified number of them successfully terminate. It is designed for batch processing tasks that run to completion.

In [ ]:
apiVersion: batch/v1
kind: Job
metadata:
  name: data-migration
spec:
  completions: 1
  parallelism: 1
  template:
    metadata:
      labels:
        app: migrator
    spec:
      containers:
      - name: migrator
        image: migrator:latest
        command: ["/bin/sh", "-c", "echo Running migration && sleep 10"]
      restartPolicy: Never

## Verify

Apply each manifest and confirm the workload behaves as expected:

In [ ]:
# Verify Deployment
kubectl get deployments
kubectl rollout status deployment/nginx-deployment

# Verify StatefulSet
kubectl get statefulsets
kubectl get pods -l app=mysql -o wide

# Verify DaemonSet
kubectl get daemonsets
kubectl get pods -l app=fluentd --show-labels

# Verify Job
kubectl get jobs
kubectl logs job/data-migration

## Key takeaways

- Deployments are the default for stateless workloads; they handle rolling updates and rollback automatically.
- StatefulSets add stable identity and ordered lifecycle, which is essential for databases and distributed systems.
- DaemonSets guarantee one pod per node, ideal for node-level agents.
- Jobs run pods to completion and are the right choice for batch and one-off tasks.
- Choosing the wrong workload type can lead to unnecessary complexity or missing features such as persistent storage or ordered startup.